# Hello, Spikes

Companion notebook for the XRDS *Hello World* column
**"Hello, Spikes: A First Spiking Neural Network for Edge Intelligence."**

Run top-to-bottom. Total wall time on a laptop: ~15 seconds after the imports.

**Setup** (skip if you already ran `uv sync` in this repo):

```bash
uv sync
uv run jupyter lab hello_spikes.ipynb
```

## Step 1 — Turn video into events

Record a short clip of yourself waving at the camera and save it as `wave.mp4`
next to this notebook. Or drop in any `.mp4` you have on hand — a hand,
a face, a swinging pendulum, anything that moves.

`eventify-dvs` walks the video pair-by-pair and emits an event whenever a
pixel's log-intensity changes by more than `c_thresh`. The output is a
NumPy structured array — nothing dense past this point.

In [ ]:
from eventify import video_to_event_stream
import numpy as np

chunks = list(video_to_event_stream(
    "wave.mp4",
    sensor_size=(128, 128),
    c_thresh=0.05,
))
events = np.concatenate(chunks)

print(f"{len(events):,} events over {events['t'].max() / 1e6:.2f} s")

A quick sanity check: bin the event timestamps and confirm that almost all
of them cluster around the moments something actually moved. Between waves,
the stream is empty.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 2))
ax.hist(events["t"] / 1e6, bins=100, color="black")
ax.set_xlabel("time (s)")
ax.set_ylabel("events per bin")
ax.set_title("Event rate over the clip")
plt.tight_layout()
plt.show()

## Step 2 — Wire events into an SNN

The workhorse of practical SNNs is the *leaky integrate-and-fire* (LIF)
neuron. Its membrane potential $v$ leaks toward a rest value and jumps
whenever an input spike arrives; when $v$ crosses a threshold, the neuron
emits a spike and resets:

$$\tau_m \frac{dv}{dt} = (v_\mathrm{rest} - v) + R\,I(t), \qquad \text{spike if } v \ge v_\mathrm{th}.$$

The pipeline has three populations:

1. **Input layer** — one neuron per pixel, driven directly by the events from Step 1.
2. **Integrator** — 8 LIF neurons, each listening to a random ~0.1% of pixels.
3. **Alarm neuron** — a single LIF cell that fires when enough of the
   integrator agrees within a short window.

Between events, the whole network is silent.

In [ ]:
from brian2 import (
    SpikeGeneratorGroup, NeuronGroup, Synapses, SpikeMonitor,
    run, mV, ms, us, second,
)
import numpy as np

# Flatten (x, y) into a single input neuron index.
idx = events["y"].astype(int) * 128 + events["x"].astype(int)
times = events["t"] * us  # eventify uses microseconds

# Brian2 requires strictly increasing spike times per source; sort just in case.
order = np.argsort(times)
pixels = SpikeGeneratorGroup(128 * 128, idx[order], times[order])

eqs = "dv/dt = (v_rest - v) / tau_m : volt"

integrator = NeuronGroup(
    N=8, model=eqs,
    threshold="v > v_th", reset="v = v_rest",
    method="exact",
    namespace={"v_rest": -70*mV, "v_th": -54*mV, "tau_m": 10*ms},
)
integrator.v = -70*mV

S = Synapses(pixels, integrator, on_pre="v_post += 0.4*mV")
S.connect(p=0.001)  # sparse random fan-in

alarm = NeuronGroup(
    N=1, model=eqs,
    threshold="v > v_th", reset="v = v_rest",
    method="exact",
    namespace={"v_rest": -70*mV, "v_th": -60*mV, "tau_m": 5*ms},
)
alarm.v = -70*mV

A = Synapses(integrator, alarm, on_pre="v_post += 4*mV")
A.connect()

print("network wired.")

## Step 3 — Run it and watch it fire

In [ ]:
M_int   = SpikeMonitor(integrator)
M_alarm = SpikeMonitor(alarm)

run(events["t"].max() * us)

print(f"integrator spikes: {M_int.num_spikes}")
print(f"alarm spikes:      {M_alarm.num_spikes}")
for t in np.asarray(M_alarm.t / ms):
    print(f"  alarm at {t:6.1f} ms")

### Raster + alarm plot

Top: every event dropped by the sensor (thinned for readability).
Middle: integrator spikes, one row per neuron.
Bottom: the alarm neuron's crisp spike(s).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True,
                         gridspec_kw={"height_ratios": [3, 2, 1]})

# Sensor events (subsampled to keep the raster readable)
sample = np.random.choice(len(events), size=min(4000, len(events)), replace=False)
axes[0].scatter(events["t"][sample] / 1e6, idx[sample], s=0.2, c="black")
axes[0].set_ylabel("pixel index")
axes[0].set_title("Sensor events (subsampled)")

# Integrator raster
axes[1].scatter(np.asarray(M_int.t / second), np.asarray(M_int.i),
                s=20, c="C0", marker="|")
axes[1].set_ylabel("integrator #")
axes[1].set_ylim(-0.5, 7.5)

# Alarm spikes
axes[2].vlines(np.asarray(M_alarm.t / second), 0, 1, color="C3", linewidth=2)
axes[2].set_ylabel("alarm")
axes[2].set_yticks([])
axes[2].set_xlabel("time (s)")

plt.tight_layout()
plt.savefig("raster_and_alarm.png", dpi=160, bbox_inches="tight")
plt.show()

## What to try next

- **Add lateral inhibition.** Give the integrator a self-`Synapses(integrator, integrator, on_pre="v_post -= 1*mV")` with `S.connect(condition='i != j')`. The alarm should now prefer coincident motion over sustained flicker.
- **Split ON and OFF.** Route positive-polarity events (`events["p"] == 1`) and negative-polarity events (`events["p"] == 0`) into two integrator pools with opposing readouts. The alarm becomes a crude direction detector.
- **Bump `c_thresh`.** Higher = fewer noise events. Watch the false-alarm rate drop and the detection latency rise.
- **Try a real event camera.** Anything that emits `(x, y, t, p)` tuples plugs into this notebook unchanged — the AEDAT/DVS/Metavision output all live behind loaders like `dv_processing` or `metavision-sdk`.